In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn
import sys
import os
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, f1_score
import seaborn as sns

from data_pipeline import get_full_pipeline


def build_mlp(in_dim, n_classes, size):
  configs = {
      "base": [256,128],
      "small": [128,64],
      "tiny": [64]
  }

  layers = []
  prev = in_dim

  for h in configs[size]:
    layers += [
        nn.Linear(prev, h),
        nn.ReLU(),
        nn.Dropout(0.3)
    ]
    prev = h

  layers.append(nn.Linear(prev, n_classes))

  return nn.Sequential(*layers)


def run_mlp_baseline(mode: int, size="base", batch_size=2048):
  (loaders, scaler, le) = get_full_pipeline(mode, batch_size=batch_size)

  train_loader, val_loader, test_loader, n_features, n_classes = loaders

  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

  model = build_mlp(n_features, n_classes, size).to(device)

  ytr = train_loader.dataset.tensors[1]
  class_counts = torch.bincount(ytr)
  class_weights = 1.0 / torch.clamp(class_counts.float(), min=1.0)

  criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
  optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

  # epochs
  def run_epoch(loader, train=True):
    model.train() if train else model.eval()

    total_correct = 0
    total_loss = 0
    total_samples = 0

    all_preds = []
    all_true = []

    with torch.set_grad_enabled(train):
      for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)

        logits = model(Xb)
        loss = criterion(logits, yb)

        if train:
          optimizer.zero_grad()
          loss.backward()
          optimizer.step()

        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == yb).sum().item()
        total_samples += yb.size(0)
        total_loss += loss.item() * yb.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(yb.cpu().numpy())

    mean_acc = total_correct / total_samples
    mean_loss = total_loss / total_samples
    macro_f1 = f1_score(all_true, all_preds, average='macro')

    return{
        "loss": mean_loss,
        "acc": mean_acc,
        "macro_f1": macro_f1,
        "preds": all_preds,
        "true": all_true
    }

  train_losses = []
  val_losses = []
  train_f1s = []
  val_f1s = []
  train_accs = []
  val_accs = []

  for epoch in range(10):
    train_epoch = run_epoch(train_loader)
    val_epoch = run_epoch(val_loader, train=False)
    train_losses.append(train_epoch["loss"])
    val_losses.append(val_epoch["loss"])
    train_f1s.append(train_epoch["macro_f1"])
    val_f1s.append(val_epoch["macro_f1"])
    train_accs.append(train_epoch["acc"])
    val_accs.append(val_epoch["acc"])
    train_loss = train_losses[-1]
    val_loss = val_losses[-1]
    train_f1 = train_f1s[-1]
    val_f1 = val_f1s[-1]
    train_acc = train_accs[-1]
    val_acc = val_accs[-1]

  test_epoch = run_epoch(test_loader, train=False)
  test_loss = test_epoch["loss"]
  test_f1 = test_epoch["macro_f1"]
  test_preds = test_epoch["preds"]
  test_trues = test_epoch["true"]

  accuracy = accuracy_score(test_trues, test_preds)
  precision, recall, f1, _ = precision_recall_fscore_support(test_trues, test_preds, average='macro', zero_division=0)
  cm = confusion_matrix(test_trues, test_preds)

  return {
      'accuracy': accuracy,
      'precision': precision,
      'recall': recall,
      'f1': f1,
      'train_loss': train_losses,
      'val_loss': val_losses,
      'confusion_matrix': cm,
      'history': val_losses
  }